In [1]:
import torch
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer, EsmForMaskedLM
from tokenizers import Tokenizer
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pickle

from scipy.spatial.distance import jensenshannon
from scipy.stats import spearmanr
from scipy.stats import rankdata


/opt/anaconda3/envs/MachLearn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from plm_compare_progen2 import *
from plm_compare_esm import *
from protein_data import *

In [3]:
with open('/Users/johnhutchens/Desktop/Practicum/Data/Wild_Dictionaries/pg2_ProGym_matrices.pickle',
           'rb') as f:
    pg_dict = pickle.load(f)

with open('/Users/johnhutchens/Desktop/Practicum/Data/Wild_Dictionaries/esm2_ProGym_matrices.pickle',
          'rb') as f:
    esm_dict = pickle.load(f)

names = list(pg_dict.keys())

In [ ]:
# example from Claude

import numpy as np
from scipy.stats import spearmanr, rankdata

# Example data
x = [1, 2, 3, 4, 5]
y = [1, 3, 2, 4, 5]

# Get Spearman correlation
corr, p_value = spearmanr(x, y)
print(f"Spearman correlation: {corr:.3f}")

# Get ranks for each list
ranks_x = rankdata(x)
ranks_y = rankdata(y)

# Find positions where ranks agree or disagree
agreements = ranks_x == ranks_y
disagreements = ~agreements

print(f"\nRanks of x: {ranks_x}")
print(f"Ranks of y: {ranks_y}")
print(f"\nPositions where ranks agree (indices): {np.where(agreements)[0]}")
print(f"Positions where ranks disagree (indices): {np.where(disagreements)[0]}")

# Show the actual differences
rank_diff = np.abs(ranks_x - ranks_y)
print(f"\nRank differences at each position: {rank_diff}")

In [5]:
pg_dict[names[3]].keys()

dict_keys(['sequence', 'DMS', 'log_probs', 'ref_log_probs', 'llr_matrix', 'spearman_DMS', 'log_probs_forward', 'ref_log_probs_forward', 'llr_matrix_forward', 'log_probs_backward', 'ref_log_probs_backward', 'llr_matrix_backward'])

In [6]:
for i in range(10):
    name = names[i]
    seq  = pg_dict[name]['sequence']
    lenseq = len(seq)
    sp_DMS = pg_dict[names[i]]['spearman_DMS'][0]

    print(i, name, lenseq, sp_DMS)

0 SDA_BACSU 44 0.3791506937886044
1 PAI1_HUMAN 402 0.3830853868977005
2 S22A1_HUMAN 553 0.5818748160195195
3 HIS7_YEAST 220 -0.019711807976736405
4 AMIE_PSEAE 346 0.6176898893049108
5 ACE2_HUMAN 805 0.09800182835485031
6 RDRP_I33A0 757 0.33020693561011744
7 CASP3_HUMAN 258 0.544815969481733
8 RL40A_YEAST 128 0.19197416052820312
9 SRC_HUMAN 536 0.39512094374115914


In [ ]:
# with open("/Users/johnhutchens/Desktop/Practicum/Data/Wild_Dictionaries/pg2_ProGym_conditional10_matrices.pickle", "wb") as f:
#     pickle.dump(pg_dict_cond_10, f)

In [15]:
for i in range(10):
    name = names[i]
    seq = pg_dict_cond_10[name]['sequence']
    lenseq = len(seq)
    # sp_val_avg = pg_dict[name]['spearman_DMS'][0]
    llr_avg = pg_dict[name]['llr_matrix']
    llr_cond = pg_dict_cond_10[name]['llr_cond']
    dms = pg_dict[name]['DMS']

    sp_val_avg, p = spearman_ignore_nan(dms, llr_avg)
    sp_val_cond, p = spearman_ignore_nan(dms, llr_cond)
    
    print(name)
    print(sp_val_avg)
    print(sp_val_cond)
    print()

SDA_BACSU
0.3791506937886044
0.2690454896802669

PAI1_HUMAN
0.3830853868977005
0.3446410428441311

S22A1_HUMAN
0.5818748160195195
0.518236668348544

HIS7_YEAST
-0.019711807976736405
-0.0049137157587255224

AMIE_PSEAE
0.6176898893049108
0.5008282183514643

ACE2_HUMAN
0.09800182835485031
0.0003394865790570384

RDRP_I33A0
0.33020693561011744
0.248633480414392

CASP3_HUMAN
0.544815969481733
0.5059314625020187

RL40A_YEAST
0.19197416052820312
0.2368130145463681

SRC_HUMAN
0.39512094374115914
0.3447548339398135



In [13]:
for i in range(10):
    name = names[i]
    seq = pg_dict_cond_10[name]['sequence']
    lenseq = len(seq)
    # sp_val_avg = pg_dict[name]['spearman_DMS'][0]
    llr_avg = pg_dict[name]['llr_matrix'][lenseq - 20:]
    llr_cond = pg_dict_cond_10[name]['llr_cond'][lenseq - 20:]
    dms = pg_dict[name]['DMS'][lenseq - 20:]

    sp_val_avg, p = spearman_ignore_nan(dms, llr_avg)
    sp_val_cond, p = spearman_ignore_nan(dms, llr_cond)
    
    print(name)
    print(sp_val_avg)
    print(sp_val_cond)
    print()

SDA_BACSU
0.48938044319241375
0.5375404209732775

PAI1_HUMAN
0.2613181842710168
0.27077515660574886

S22A1_HUMAN
0.061139724383437476
0.07827506120143118

HIS7_YEAST
-0.3916083916083916
-0.2447552447552448

AMIE_PSEAE
-0.36843754109271276
-0.3940118323736629

ACE2_HUMAN
nan
nan

RDRP_I33A0
0.3601004474176912
0.32667250239483475

CASP3_HUMAN
0.5248818563822493
0.5928876918664822

RL40A_YEAST
nan
nan

SRC_HUMAN
0.2961067853170189
0.2507230255839822

